<div style="font-family:system-ui;border-left:6px solid #B8721A;padding:6px 16px">
<div style="font-size:12px;letter-spacing:.12em;text-transform:uppercase;color:#566273">The AI-Ready Organisation · Lab 3 · Module 3 · ≈35 min</div>
<div style="font-size:26px;font-weight:800;margin:6px 0">Audit a real transactional dataset</div>
<div style="font-size:15px"><b>You will:</b> profile two years of online retail transactions, produce a data quality scorecard, and estimate how much of the data a customer-level AI model could actually use.</div>
<div style="font-size:13px;color:#566273;margin-top:6px">Dataset: UCI Online Retail II (about one million transactions from a UK online retailer, 2009–2011)</div>
</div>

**How to use this notebook** · No coding needed.
1. In the menu choose **Runtime → Run all** (in Kaggle: **Run All**). The first run takes 1–3 minutes.
2. Change values in the **form fields** on the right of each cell (sliders, dropdowns). The cell reruns when you change a value; if it does not, press the play button.
3. Read the plain-English findings under each chart and record your answers in the final cell.

If the real dataset cannot be downloaded (for example, without a Kaggle sign-in), the notebook automatically switches to a **realistic simulated copy with the same structure** and tells you so. The lessons still hold.

In [ ]:
#@title Setup (run first) { display-mode: "form" }
import warnings, os, io, zipfile, urllib.request
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
plt.rcParams.update({"figure.figsize": (9, 4.4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 11, "axes.titleweight": "bold"})
BUILD, BUY, BLEND, INK, GOOD, BAD = "#B8721A", "#13807B", "#4255B0", "#111822", "#2D7A4C", "#B03A3A"
pd.set_option("display.max_columns", 30)

def say(text):
    display(Markdown(text))

def scorecard(title, rows):
    """rows: list of (label, value, note)"""
    body = "".join(f"<tr><td style='padding:6px 12px'>{a}</td><td style='padding:6px 12px;font-weight:700;text-align:right'>{b}</td>"
                   f"<td style='padding:6px 12px;color:#566273'>{c}</td></tr>" for a, b, c in rows)
    display(HTML(f"<div style='font-family:system-ui;margin:8px 0'><div style='font-weight:700;font-size:15px;margin-bottom:4px'>{title}</div>"
                 f"<table style='border-collapse:collapse;font-size:14px'>{body}</table></div>"))

def try_kagglehub(handle, filename):
    """Download a public Kaggle dataset. Returns a file path or None."""
    try:
        try:
            import kagglehub
        except ImportError:
            os.system("pip -q install kagglehub")
            import kagglehub
        path = kagglehub.dataset_download(handle)
        for root, _, files in os.walk(path):
            if filename in files:
                return os.path.join(root, filename)
    except Exception as e:
        print(f"Kaggle download unavailable ({type(e).__name__}).")
    return None

RNG = np.random.default_rng(42)
DATA_NOTE = ""

## 1 · Load the transactions
The source file is an Excel workbook of about 45 MB, so this cell can take two to three minutes in Colab.

In [ ]:
#@title Load data { display-mode: "form" }
try:
    raw = urllib.request.urlopen("https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip", timeout=60).read()
    z = zipfile.ZipFile(io.BytesIO(raw)); name = [n for n in z.namelist() if n.endswith(".xlsx")][0]
    sheets = pd.read_excel(z.open(name), sheet_name=None)
    df = pd.concat(sheets.values(), ignore_index=True)
    DATA_NOTE = "Real UCI Online Retail II data."
except Exception:
    n = 120_000
    inv = RNG.integers(489000, 581600, n).astype(str)
    cancel = RNG.random(n) < .018
    inv = np.where(cancel, "C" + inv, inv)
    codes = np.where(RNG.random(n) < .006, RNG.choice(["POST", "M", "D", "BANK CHARGES", "DOT"], n), RNG.integers(10000, 90000, n).astype(str))
    qty = np.where(cancel, -RNG.integers(1, 24, n), RNG.integers(1, 48, n))
    qty = np.where(RNG.random(n) < .004, -RNG.integers(1, 50, n), qty)
    price = np.round(RNG.gamma(2, 1.8, n), 2); price[RNG.random(n) < .006] = 0
    cust = RNG.integers(12346, 18288, n).astype(float); cust[RNG.random(n) < .23] = np.nan
    desc = RNG.choice(["WHITE HANGING HEART T-LIGHT HOLDER", "REGENCY CAKESTAND 3 TIER", "JUMBO BAG RED RETROSPOT", "PARTY BUNTING", "LUNCH BAG RED RETROSPOT"], n).astype(object)
    desc[RNG.random(n) < .004] = np.nan
    dates = pd.to_datetime("2009-12-01") + pd.to_timedelta(RNG.integers(0, 739, n), "D") + pd.to_timedelta(RNG.integers(8*60, 20*60, n), "m")
    df = pd.DataFrame({"Invoice": inv, "StockCode": codes, "Description": desc, "Quantity": qty, "InvoiceDate": dates,
                       "Price": price, "Customer ID": cust, "Country": RNG.choice(["United Kingdom", "Germany", "France", "EIRE", "Netherlands"], n, p=[.9, .03, .03, .02, .02])})
    df = pd.concat([df, df.sample(frac=.02, random_state=1)], ignore_index=True)
    DATA_NOTE = "Simulated copy (UCI download unavailable). Same columns, with similar quality issues injected."
df["Invoice"] = df["Invoice"].astype(str); df["StockCode"] = df["StockCode"].astype(str)
say(f"**{DATA_NOTE}** {len(df):,} rows from {df.InvoiceDate.min():%b %Y} to {df.InvoiceDate.max():%b %Y}.")
display(df.head())

## 2 · Run the automated quality checks

In [ ]:
#@title Data quality scorecard { display-mode: "form" }
N = len(df)
checks = {
    "Missing customer ID": df["Customer ID"].isna(),
    "Missing product description": df["Description"].isna(),
    "Cancelled invoice": df["Invoice"].str.startswith("C"),
    "Zero or negative quantity": df["Quantity"] <= 0,
    "Zero or negative price": df["Price"] <= 0,
    "Exact duplicate row": df.duplicated(keep="first"),
    "Non-product code (postage, fees, adjustments)": df["StockCode"].str.fullmatch(r"[A-Za-z ]+") .fillna(False),
}
sc = pd.DataFrame({"Rows affected": {k: int(v.sum()) for k, v in checks.items()}})
sc["% of rows"] = (sc["Rows affected"] / N * 100).round(2)
sc["Severity for a customer model"] = ["High", "Low", "Medium", "Medium", "Medium", "Low", "Low"]
display(sc.sort_values("% of rows", ascending=False))
fig, ax = plt.subplots(figsize=(9, 3.8))
s2 = sc.sort_values("% of rows")
ax.barh(s2.index, s2["% of rows"], color=[BAD if v == "High" else BUILD if v == "Medium" else "#9AA5B1" for v in s2["Severity for a customer model"]])
ax.set(title="Share of rows affected by each issue", xlabel="% of rows"); plt.show()
bad = np.zeros(N, dtype=bool)
for v in checks.values(): bad |= v.to_numpy()
usable = 1 - bad.mean()
say(f"### Only **{usable:.0%}** of rows pass every check and can feed a customer-level model.\nThe largest single issue is **{sc['% of rows'].idxmax().lower()}** ({sc['% of rows'].max():.1f}% of rows).")

## 3 · Is coverage consistent over time?
Gaps or sudden drops usually point to system migrations or broken feeds, which models quietly learn as if they were real behaviour.

In [ ]:
#@title Monthly coverage { display-mode: "form" }
m = df.set_index("InvoiceDate").resample("MS").agg({"Invoice": "count", "Customer ID": lambda s: s.isna().mean()*100})
m.columns = ["Rows", "% missing customer ID"]
fig, ax = plt.subplots(); ax.bar(m.index, m.Rows, width=20, color=BLEND, alpha=.8); ax.set(title="Rows per month", ylabel="Rows")
ax2 = ax.twinx(); ax2.plot(m.index, m["% missing customer ID"], color=BAD, lw=2, marker="o", ms=3); ax2.set_ylabel("% missing customer ID", color=BAD); ax2.grid(False); plt.show()
say(f"Missing customer IDs range from **{m['% missing customer ID'].min():.0f}%** to **{m['% missing customer ID'].max():.0f}%** by month.")

## 4 · Score this domain on the readiness radar
Four dimensions are measured from the data. **Lineage** and **governance** cannot be measured from a file, so rate them yourself from what you know about where the data came from.

In [ ]:
#@title Readiness scores for Simulator 3.2 { display-mode: "form" }
lineage_score = 2 #@param {type:"slider", min:1, max:5, step:1}
governance_score = 2 #@param {type:"slider", min:1, max:5, step:1}
to5 = lambda pct_good: int(np.clip(round(1 + 4*pct_good), 1, 5))
months_covered = m.Rows.gt(0).sum(); expected = len(m)
scores = {
    "Availability": to5(months_covered/expected),
    "Quality": to5(usable),
    "Accessibility": 4,
    "Lineage and metadata": lineage_score,
    "Governance and privacy": governance_score,
    "Unstructured readiness": to5(1 - checks["Missing product description"].mean()) - 2,
}
scores["Unstructured readiness"] = max(1, scores["Unstructured readiness"])
ang = np.linspace(0, 2*np.pi, len(scores), endpoint=False).tolist(); vals = list(scores.values())
fig = plt.figure(figsize=(5.5, 5.5)); ax = fig.add_subplot(polar=True)
ax.plot(ang + ang[:1], vals + vals[:1], color=BUY, lw=2.5); ax.fill(ang + ang[:1], vals + vals[:1], color=BUY, alpha=.2)
ax.set_xticks(ang); ax.set_xticklabels(list(scores), fontsize=9); ax.set_ylim(0, 5); ax.set_yticks([1, 2, 3, 4, 5]); plt.show()
display(pd.Series(scores, name="Score (1–5)").to_frame())
weakest = min(scores, key=scores.get)
say(f"Average **{np.mean(vals):.1f} / 5**. Fix first: **{weakest.lower()}**. Enter these scores in Simulator 3.2 on the course page.  \n_Accessibility is set to 4 because the file is openly published; change it for your own data._")

## Your reflection → capstone §3
Answer in the form fields, then run the cell. Copy the summary into §3 of your business case on the course page.

In [ ]:
#@title Record your answers { display-mode: "form" }
q1 = "" #@param {type:"string"}
q2 = "" #@param {type:"string"}
q3 = "" #@param {type:"string"}
answers = [("The share of usable rows and what it means for my use case:", q1), ("The first data fix I would fund, and who should own it:", q2), ("One equivalent issue I suspect in my own organisation's data:", q3)]
say("### Lab 3 reflection")
for q, a in answers:
    say(f"**{q}**  \n{a if a.strip() else '_(not answered yet)_'}")